In [4]:
# ============================================================
# statistical_validation.ipynb
# Full Statistical Validation — All Phases
# Author  : [Your Name]
# Date    : 2026-05-14
# ============================================================


# %% [markdown]
# ## 0. Libraries and Data Loading

# %%
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Phase 1 aggregate values (from dbt-labs benchmark re-analysis)
# Total records: 5,170 → 2,585 per method
P1_SL_ACC,  P1_SL_N  = 0.754, 2585
P1_SQL_ACC, P1_SQL_N = 0.629, 2585

df_p2 = pd.read_csv("./results_phase2_bird_v2.csv")
df_p3 = pd.read_csv("./results_phase3_v2.csv")

print("Data loaded.")
print(f"  Phase 1 : aggregate (n={P1_SL_N} per method)")
print(f"  Phase 2 : {len(df_p2)} records")
print(f"  Phase 3 : {len(df_p3)} records")


# %% [markdown]
# ## 1. Helper Functions

# %%
def cohens_d(p1: float, p2: float) -> float:
    """Cohen's d for two proportions using pooled SD."""
    pooled = np.sqrt((p1*(1-p1) + p2*(1-p2)) / 2)
    return (p1 - p2) / pooled if pooled > 0 else 0.0

def interpret_d(d: float) -> str:
    a = abs(d)
    if a < 0.2:   return "negligible"
    elif a < 0.5: return "small"
    elif a < 0.8: return "medium"
    else:         return "large"

def chi_square(sl_c, sl_n, sql_c, sql_n):
    """Chi-square test for two proportions."""
    table = [[sl_c, sl_n-sl_c], [sql_c, sql_n-sql_c]]
    chi2, p, dof, _ = chi2_contingency(table)
    return chi2, p, dof

def mcnemar(df, qid='question_id', method='method',
            correct='is_correct', sl='semantic_layer', sql='text_to_sql'):
    """McNemar's test on question-level mean accuracy."""
    q_mean = df.groupby([qid, method])[correct].mean().unstack()
    b = int((q_mean[sl]  > q_mean[sql]).sum())
    c = int((q_mean[sql] > q_mean[sl]).sum())
    if b + c == 0:
        return None, None, b, c
    mc = (abs(b-c) - 1)**2 / (b+c)
    p  = 1 - stats.chi2.cdf(mc, df=1)
    return mc, p, b, c

def bootstrap_ci(data, n_boot=5000, ci=0.95):
    """Bootstrap confidence interval for mean."""
    boots = [np.mean(np.random.choice(data, len(data), replace=True))
             for _ in range(n_boot)]
    lo = np.percentile(boots, (1-ci)/2*100)
    hi = np.percentile(boots, (1+ci)/2*100)
    return lo, hi

print("Helper functions defined.")


# %% [markdown]
# ## 2. Chi-square Test — Overall Accuracy per Phase

# %%
print("=" * 65)
print("[1] Chi-square Test — Overall Accuracy per Phase")
print("=" * 65)

phases = {
    "Phase 1 (ACME, n=2585)": {
        "sl" : (int(P1_SL_ACC  * P1_SL_N),  P1_SL_N),
        "sql": (int(P1_SQL_ACC * P1_SQL_N), P1_SQL_N),
    },
    "Phase 2 (BIRD, n=106)": {
        "sl" : (int(df_p2[df_p2['method']=='semantic_layer']['is_correct'].sum()),
                len(df_p2[df_p2['method']=='semantic_layer'])),
        "sql": (int(df_p2[df_p2['method']=='text_to_sql']['is_correct'].sum()),
                len(df_p2[df_p2['method']=='text_to_sql'])),
    },
    "Phase 3 (Credit, n=300)": {
        "sl" : (int(df_p3[df_p3['method']=='semantic_layer']['is_correct'].sum()),
                len(df_p3[df_p3['method']=='semantic_layer'])),
        "sql": (int(df_p3[df_p3['method']=='text_to_sql']['is_correct'].sum()),
                len(df_p3[df_p3['method']=='text_to_sql'])),
    },
}

print(f"\n  {'Phase':<28} {'SL':>7} {'SQL':>7} {'Δ':>7} "
      f"{'χ²':>8} {'p':>8} {'d':>8} {'Interp.'}")
print("  " + "-" * 82)

for phase, data in phases.items():
    sl_c,  sl_n  = data['sl']
    sql_c, sql_n = data['sql']
    sl_r,  sql_r = sl_c/sl_n, sql_c/sql_n
    chi2, p, _   = chi_square(sl_c, sl_n, sql_c, sql_n)
    d            = cohens_d(sl_r, sql_r)
    sig          = '***' if p<.001 else ('*' if p<.05 else 'n.s.')
    print(f"  {phase:<28} {sl_r:>7.3f} {sql_r:>7.3f} {sl_r-sql_r:>+7.3f} "
          f"{chi2:>8.3f} {p:>8.4f} {d:>+8.3f} {interpret_d(d)} {sig}")


# %% [markdown]
# ## 3. McNemar's Test — Paired Comparison

# %%
print("\n" + "=" * 65)
print("[2] McNemar's Test — Paired Question-level Comparison")
print("=" * 65)

print(f"\n  {'Phase':<12} {'SL better':>12} {'SQL better':>12} {'χ²':>8} {'p':>8} {'Sig.'}")
print("  " + "-" * 58)

for phase, df in [("Phase 2", df_p2), ("Phase 3", df_p3)]:
    mc, p, b, c = mcnemar(df)
    if mc is not None:
        sig = '***' if p<.001 else ('*' if p<.05 else 'n.s.')
        print(f"  {phase:<12} {b:>12} {c:>12} {mc:>8.3f} {p:>8.4f}  {sig}")
    else:
        print(f"  {phase:<12} {b:>12} {c:>12} {'N/A':>8} {'N/A':>8}")


# %% [markdown]
# ## 4. One-way ANOVA — Accuracy by Difficulty

# %%
print("\n" + "=" * 65)
print("[3] One-way ANOVA — Accuracy by Difficulty Level")
print("=" * 65)

diffs = ['simple', 'moderate', 'challenging']

for phase, df in [("Phase 2", df_p2), ("Phase 3", df_p3)]:
    print(f"\n  {phase}")
    for method in ['semantic_layer', 'text_to_sql']:
        label  = 'SL ' if method=='semantic_layer' else 'SQL'
        groups = [df[(df['method']==method) &
                     (df['difficulty']==d)]['is_correct'].astype(float).values
                  for d in diffs]
        means  = [g.mean() for g in groups]
        f, p   = f_oneway(*groups)
        sig    = '***' if p<.001 else ('*' if p<.05 else 'n.s.')
        print(f"    [{label}] simple={means[0]:.3f}  moderate={means[1]:.3f}  "
              f"challenging={means[2]:.3f} | F={f:.3f}, p={p:.4f} {sig}")

    # Kruskal-Wallis (non-parametric)
    print(f"    Kruskal-Wallis:")
    for method in ['semantic_layer', 'text_to_sql']:
        label  = 'SL ' if method=='semantic_layer' else 'SQL'
        groups = [df[(df['method']==method) &
                     (df['difficulty']==d)]['is_correct'].astype(float).values
                  for d in diffs]
        h, p   = kruskal(*groups)
        sig    = '***' if p<.001 else ('*' if p<.05 else 'n.s.')
        print(f"    [{label}] H={h:.3f}, p={p:.4f} {sig}")


# %% [markdown]
# ## 5. Bootstrap 95% Confidence Intervals

# %%
print("\n" + "=" * 65)
print("[4] Bootstrap 95% Confidence Intervals (n_boot=5000)")
print("=" * 65)

np.random.seed(42)
print(f"\n  {'Phase':<12} {'Method':<18} {'Mean':>8} {'95% CI'}")
print("  " + "-" * 55)

for phase, df in [("Phase 2", df_p2), ("Phase 3", df_p3)]:
    for method in ['semantic_layer', 'text_to_sql']:
        label = 'Semantic Layer' if method=='semantic_layer' else 'Text-to-SQL   '
        data  = df[df['method']==method]['is_correct'].astype(float).values
        mean  = data.mean()
        lo, hi = bootstrap_ci(data)
        print(f"  {phase:<12} {label:<18} {mean:>8.3f} [{lo:.3f}, {hi:.3f}]")


# %% [markdown]
# ## 6. Effect Size Summary

# %%
print("\n" + "=" * 65)
print("[5] Effect Size Summary — Cohen's d")
print("=" * 65)
print(f"\n  {'Comparison':<48} {'d':>8}  Interpretation")
print("  " + "-" * 72)

def get_acc(df, method):
    return df[df['method']==method]['is_correct'].mean()

comparisons = [
    ("Phase 1: SL vs SQL (overall)",
     P1_SL_ACC, P1_SQL_ACC),
    ("Phase 2: SL vs SQL (overall)",
     get_acc(df_p2,'semantic_layer'), get_acc(df_p2,'text_to_sql')),
    ("Phase 3: SL vs SQL (overall)",
     get_acc(df_p3,'semantic_layer'), get_acc(df_p3,'text_to_sql')),
    ("Phase 3: SL vs SQL (default_risk)",
     df_p3[(df_p3['method']=='semantic_layer') & (df_p3['category']=='default_risk')]['is_correct'].mean(),
     df_p3[(df_p3['method']=='text_to_sql')    & (df_p3['category']=='default_risk')]['is_correct'].mean()),
    ("Phase 3: SL vs SQL (regional_risk)",
     df_p3[(df_p3['method']=='semantic_layer') & (df_p3['category']=='regional_risk')]['is_correct'].mean(),
     df_p3[(df_p3['method']=='text_to_sql')    & (df_p3['category']=='regional_risk')]['is_correct'].mean()),
]

for label, p1, p2 in comparisons:
    d = cohens_d(p1, p2)
    print(f"  {label:<48} {d:>+8.3f}  {interpret_d(d)}")

# RQ2 exposure note
print(f"\n  {'RQ2: Sensitive columns exposed (SQL=15, SL=0)':<48} "
      f"{'100% structural elimination (not applicable to Cohen d)'}")


# %% [markdown]
# ## 7. Category-level Effect Sizes (Phase 3)

# %%
print("\n" + "=" * 65)
print("[6] Category-level Effect Sizes — Phase 3")
print("=" * 65)

by_cat = df_p3.groupby(['category','method'])['is_correct'].mean().unstack()
print(f"\n  {'Category':<25} {'SL':>8} {'SQL':>8} {'d':>8}  Interpretation")
print("  " + "-" * 60)

for cat in by_cat.index:
    sl_acc  = by_cat.loc[cat,'semantic_layer']
    sql_acc = by_cat.loc[cat,'text_to_sql']
    d = cohens_d(sl_acc, sql_acc)
    print(f"  {cat:<25} {sl_acc:>8.3f} {sql_acc:>8.3f} {d:>+8.3f}  {interpret_d(d)}")


# %% [markdown]
# ## 8. Export Statistical Results to CSV

# %%
import os

# -------------------------------------------------------
# 8-1. Accuracy comparison summary
# -------------------------------------------------------
rows_acc = []
for phase, sl_acc, sl_n, sql_acc, sql_n in [
    ("Phase 1 (ACME)",        P1_SL_ACC,  P1_SL_N,  P1_SQL_ACC, P1_SQL_N),
    ("Phase 2 (BIRD)",
     df_p2[df_p2['method']=='semantic_layer']['is_correct'].mean(), len(df_p2[df_p2['method']=='semantic_layer']),
     df_p2[df_p2['method']=='text_to_sql']['is_correct'].mean(),    len(df_p2[df_p2['method']=='text_to_sql'])),
    ("Phase 3 (Credit)",
     df_p3[df_p3['method']=='semantic_layer']['is_correct'].mean(), len(df_p3[df_p3['method']=='semantic_layer']),
     df_p3[df_p3['method']=='text_to_sql']['is_correct'].mean(),    len(df_p3[df_p3['method']=='text_to_sql'])),
]:
    sl_c  = int(sl_acc  * sl_n)
    sql_c = int(sql_acc * sql_n)
    chi2, p, _ = chi_square(sl_c, sl_n, sql_c, sql_n)
    d   = cohens_d(sl_acc, sql_acc)
    sig = '***' if p<.001 else ('*' if p<.05 else 'n.s.')
    rows_acc.append({
        'phase'         : phase,
        'sl_accuracy'   : round(sl_acc,  4),
        'sql_accuracy'  : round(sql_acc, 4),
        'sl_n'          : sl_n,
        'sql_n'         : sql_n,
        'delta'         : round(sl_acc - sql_acc, 4),
        'chi2'          : round(chi2, 4),
        'p_value'       : round(p,    4),
        'significance'  : sig,
        'cohens_d'      : round(d, 4),
        'd_interpretation': interpret_d(d),
    })

df_acc_summary = pd.DataFrame(rows_acc)

# -------------------------------------------------------
# 8-2. ANOVA summary
# -------------------------------------------------------
rows_anova = []
for phase, df in [("Phase 2 (BIRD)", df_p2), ("Phase 3 (Credit)", df_p3)]:
    for method in ['semantic_layer', 'text_to_sql']:
        groups = [df[(df['method']==method) &
                     (df['difficulty']==d)]['is_correct'].astype(float).values
                  for d in ['simple','moderate','challenging']]
        means = [g.mean() for g in groups]
        f, p  = f_oneway(*groups)
        rows_anova.append({
            'phase'            : phase,
            'method'           : method,
            'simple_accuracy'  : round(means[0], 4),
            'moderate_accuracy': round(means[1], 4),
            'challenging_accuracy': round(means[2], 4),
            'F_statistic'      : round(f, 4),
            'p_value'          : round(p, 4),
            'significance'     : '***' if p<.001 else ('*' if p<.05 else 'n.s.'),
        })

df_anova_summary = pd.DataFrame(rows_anova)

# -------------------------------------------------------
# 8-3. Effect size summary
# -------------------------------------------------------
rows_d = []
comparisons_export = [
    ("Phase 1: SL vs SQL (overall)",       P1_SL_ACC,  P1_SQL_ACC),
    ("Phase 2: SL vs SQL (overall)",
     get_acc(df_p2,'semantic_layer'), get_acc(df_p2,'text_to_sql')),
    ("Phase 3: SL vs SQL (overall)",
     get_acc(df_p3,'semantic_layer'), get_acc(df_p3,'text_to_sql')),
    ("Phase 3: SL vs SQL (default_risk)",
     df_p3[(df_p3['method']=='semantic_layer') & (df_p3['category']=='default_risk')]['is_correct'].mean(),
     df_p3[(df_p3['method']=='text_to_sql')    & (df_p3['category']=='default_risk')]['is_correct'].mean()),
    ("Phase 3: SL vs SQL (regional_risk)",
     df_p3[(df_p3['method']=='semantic_layer') & (df_p3['category']=='regional_risk')]['is_correct'].mean(),
     df_p3[(df_p3['method']=='text_to_sql')    & (df_p3['category']=='regional_risk')]['is_correct'].mean()),
]

for label, p1, p2 in comparisons_export:
    d = cohens_d(p1, p2)
    rows_d.append({
        'comparison'      : label,
        'sl_accuracy'     : round(p1, 4),
        'sql_accuracy'    : round(p2, 4),
        'cohens_d'        : round(d,  4),
        'd_interpretation': interpret_d(d),
    })

df_d_summary = pd.DataFrame(rows_d)

# -------------------------------------------------------
# 8-4. Bootstrap CI summary
# -------------------------------------------------------
np.random.seed(42)
rows_ci = []
for phase, df in [("Phase 2 (BIRD)", df_p2), ("Phase 3 (Credit)", df_p3)]:
    for method in ['semantic_layer', 'text_to_sql']:
        data   = df[df['method']==method]['is_correct'].astype(float).values
        mean   = data.mean()
        lo, hi = bootstrap_ci(data)
        rows_ci.append({
            'phase'  : phase,
            'method' : method,
            'mean'   : round(mean, 4),
            'ci_lower': round(lo,  4),
            'ci_upper': round(hi,  4),
        })

df_ci_summary = pd.DataFrame(rows_ci)

# -------------------------------------------------------
# 8-5. RQ2 exposure summary
# -------------------------------------------------------
df_rq2_summary = pd.DataFrame([
    {'metric': 'Sensitive columns exposed (distinct)', 'text_to_sql': 15, 'semantic_layer': 0, 'reduction_pct': 100.0},
    {'metric': 'Internal codes exposed (distinct)',    'text_to_sql':  0, 'semantic_layer': 0, 'reduction_pct': None},
    {'metric': 'Schema context size (chars)',          'text_to_sql': 976,'semantic_layer': 3555,'reduction_pct': -264.2},
])

# -------------------------------------------------------
# Save all to CSV
# -------------------------------------------------------
df_acc_summary.to_csv('./results_statistical_accuracy.csv',   index=False, encoding='utf-8-sig')
df_anova_summary.to_csv('./results_statistical_anova.csv',    index=False, encoding='utf-8-sig')
df_d_summary.to_csv('./results_statistical_effectsize.csv',   index=False, encoding='utf-8-sig')
df_ci_summary.to_csv('./results_statistical_bootstrap_ci.csv',index=False, encoding='utf-8-sig')
df_rq2_summary.to_csv('./results_rq2_exposure.csv',           index=False, encoding='utf-8-sig')

print("Statistical CSVs saved:")
csvs = ['results_statistical_accuracy.csv', 'results_statistical_anova.csv',
        'results_statistical_effectsize.csv', 'results_statistical_bootstrap_ci.csv',
        'results_rq2_exposure.csv']
for f in csvs:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

print("\n[Accuracy Summary]")
print(df_acc_summary.to_string(index=False))
print("\n[ANOVA Summary]")
print(df_anova_summary.to_string(index=False))
print("\n[Effect Size Summary]")
print(df_d_summary.to_string(index=False))
print("\n[Bootstrap CI Summary]")
print(df_ci_summary.to_string(index=False))
print("\n[RQ2 Exposure Summary]")
print(df_rq2_summary.to_string(index=False))


# %% [markdown]
# ## 9. Paper-ready Summary Table

# %%
print("\n" + "=" * 65)
print("PAPER-READY STATISTICAL SUMMARY")
print("=" * 65)

print("""
  RQ1 — Accuracy
  ┌──────────┬──────────┬──────────┬────────┬───────────────────────────┐
  │ Phase    │ SL Acc.  │ SQL Acc. │ Sig.   │ Key Finding               │
  ├──────────┼──────────┼──────────┼────────┼───────────────────────────┤
  │ Phase 1  │  75.4%   │  62.9%   │ ***    │ SL significantly better   │
  │ Phase 2  │  42.5%   │  38.7%   │ n.s.   │ SL equivalent to SQL      │
  │ Phase 3  │  38.3%   │  39.0%   │ n.s.   │ SL equivalent to SQL      │
  ├──────────┼──────────┼──────────┼────────┼───────────────────────────┤
  │ ANOVA    │ —        │ —        │ ***    │ Jagged Frontier confirmed  │
  │ default_ │  40.0%   │  20.0%   │ d=.447 │ SL small-medium (credit)  │
  └──────────┴──────────┴──────────┴────────┴───────────────────────────┘

  RQ2 — Security
  ┌────────────────────────────┬──────┬──────┬─────────────────────────┐
  │ Metric                     │ SQL  │ SL   │ Result                  │
  ├────────────────────────────┼──────┼──────┼─────────────────────────┤
  │ Sensitive columns exposed  │  15  │   0  │ 100% structural elim.   │
  │ (schema-level, per prompt) │      │      │ Information Hiding impl. │
  └────────────────────────────┴──────┴──────┴─────────────────────────┘

  RQ1-sub — Latency
  ┌──────────┬─────────────────────┬─────────────────────┐
  │ Phase    │ SL mean (ms)        │ SQL mean (ms)        │
  ├──────────┼─────────────────────┼─────────────────────┤
  │ Phase 2  │ 1,397               │ 1,710                │
  │ Phase 3  │ 1,481               │ 3,393                │
  └──────────┴─────────────────────┴─────────────────────┘
  → SL shows lower or equal latency — no efficiency trade-off
""")

Data loaded.
  Phase 1 : aggregate (n=2585 per method)
  Phase 2 : 212 records
  Phase 3 : 600 records
Helper functions defined.
[1] Chi-square Test — Overall Accuracy per Phase

  Phase                             SL     SQL       Δ       χ²        p        d Interp.
  ----------------------------------------------------------------------------------
  Phase 1 (ACME, n=2585)         0.754   0.629  +0.125   94.560   0.0000   +0.274 small ***
  Phase 2 (BIRD, n=106)          0.425   0.387  +0.038    0.176   0.6748   +0.077 negligible n.s.
  Phase 3 (Credit, n=300)        0.383   0.390  -0.007    0.007   0.9332   -0.014 negligible n.s.

[2] McNemar's Test — Paired Question-level Comparison

  Phase           SL better   SQL better       χ²        p Sig.
  ----------------------------------------------------------
  Phase 2                10            6    0.562   0.4533  n.s.
  Phase 3                 9            8    0.000   1.0000  n.s.

[3] One-way ANOVA — Accuracy by Difficulty Lev